# 03 — Explore past runs (no models, no LM Studio)

Lightweight viewer for everything cached under `predictions/<slug>/scores.json`.
Auto-discovers all runs (trained parsers, 0-shot LLMs, k-shot LLMs, archived
variants) and renders the comparison without importing `latinbench` or touching
any external service. Add a new run = drop a `predictions/<new_slug>/scores.json`
on disk and re-run this notebook.

For the live bench (predicting + scoring): [`02_compare_models.ipynb`](02_compare_models.ipynb).\
For the findings writeup: [`../docs/01_findings.md`](../docs/01_findings.md).

## 1. Load all cached scores

The slug parser pulls `(family, shots, seed)` out of names like
`qwen3-vl-8b-instruct-mlx-2shot-s7`, so few-shot variants appear in the same
tables as their 0-shot siblings, broken out by `shots` rather than collapsed
into a different system name.

In [1]:
import json
import re
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PREDICTIONS_DIR = REPO_ROOT / "predictions"

# Slug format: <family>[-<k>shot[-s<seed>]] — see LMStudioModel.__init__.
_SLUG_RE = re.compile(r"^(.+?)(?:-(\d+)shot(?:-s(\d+))?)?$")

def parse_slug(slug: str) -> tuple[str, int, int]:
    """(family, shots, seed). Non-LLM slugs (no -Nshot suffix) get shots=0, seed=0."""
    m = _SLUG_RE.match(slug)
    family, shots, seed = m.group(1), m.group(2), m.group(3)
    return family, int(shots) if shots else 0, int(seed) if seed else 0

rows = []
for scores_file in sorted(PREDICTIONS_DIR.glob("*/scores.json")):
    slug = scores_file.parent.name
    family, shots, seed = parse_slug(slug)
    scores = json.loads(scores_file.read_text())
    for split, by_metric in scores.items():
        for metric, vals in by_metric.items():
            rows.append({
                "slug": slug,
                "family": family,
                "shots": shots,
                "seed": seed,
                "split": split,
                "metric": metric,
                "F1": vals["F1"],
                "P": vals["P"],
                "R": vals["R"],
            })

df = pd.DataFrame(rows)
print(f"Loaded {len(df)} rows from {df['slug'].nunique()} cached runs:")
for slug, grp in df.groupby("slug"):
    fam, sh, sd = parse_slug(slug)
    tag = f"k={sh}" + (f", seed={sd}" if sd else "")
    print(f"  {slug:50}  family={fam:35}  {tag}")

Loaded 260 rows from 10 cached runs:
  latin-perseus-ud-2.15-241121                        family=latin-perseus-ud-2.15-241121         k=0
  latin-perseus-ud-2.17-251125                        family=latin-perseus-ud-2.17-251125         k=0
  latinpipe                                           family=latinpipe                            k=0
  my_model                                            family=my_model                             k=0
  qwen3-0.6b                                          family=qwen3-0.6b                           k=0
  qwen3-0.6b-mlx                                      family=qwen3-0.6b-mlx                       k=0
  qwen3-0.6b-mlx-2shot                                family=qwen3-0.6b-mlx                       k=2
  qwen3-0.6b-mlx.pre-repair                           family=qwen3-0.6b-mlx.pre-repair            k=0
  qwen3-vl-8b-instruct-mlx                            family=qwen3-vl-8b-instruct-mlx             k=0
  qwen3-vl-8b-instruct-mlx-2shot             

## 2. Headline LAS

LAS (labeled attachment score) is the primary parsing metric. Rows are
`(family, k-shot)`; columns are the two test splits.

In [2]:
las = df[df["metric"] == "LAS"]
las_table = las.pivot_table(
    index=["family", "shots"],
    columns="split",
    values="F1",
).round(2)
las_table

split                               poetry  prose
family                       shots               
latin-perseus-ud-2.15-241121 0       50.78  57.68
latin-perseus-ud-2.17-251125 0       61.19  62.43
latinpipe                    0       72.27  75.06
my_model                     0        1.14   0.60
qwen3-0.6b                   0        1.74   0.88
qwen3-0.6b-mlx               0        2.67   1.62
                             2        2.06   1.32
qwen3-0.6b-mlx.pre-repair    0        1.94   1.09
qwen3-vl-8b-instruct-mlx     0       18.21  17.80
                             2       18.41  20.16

## 3. Few-shot vs zero-shot delta

For every family that has both a 0-shot and a k-shot cache, what's the LAS
change? Positive = few-shot helps, negative = hurts.

In [3]:
# Find families with multiple `shots` values cached.
have_shots = las.groupby("family")["shots"].nunique()
few_shot_families = have_shots[have_shots > 1].index.tolist()

deltas = []
for family in few_shot_families:
    sub = las[las["family"] == family].pivot_table(
        index="shots", columns="split", values="F1",
    )
    if 0 not in sub.index:
        continue
    for k in sub.index.drop(0):
        for split in sub.columns:
            zs = sub.loc[0, split]
            ks = sub.loc[k, split]
            deltas.append({
                "family": family,
                "split": split,
                "k": int(k),
                "0-shot LAS": round(zs, 2),
                f"k-shot LAS": round(ks, 2),
                "Δ": round(ks - zs, 2),
            })

pd.DataFrame(deltas)

,family,split,k,0-shot LAS,k-shot LAS,Δ
0,qwen3-0.6b-mlx,poetry,2,2.67,2.06,-0.61
1,qwen3-0.6b-mlx,prose,2,1.62,1.32,-0.30
2,qwen3-vl-8b-instruct-mlx,poetry,2,18.21,18.41,0.20
3,qwen3-vl-8b-instruct-mlx,prose,2,17.80,20.16,2.36


## 4. UAS / LAS / CLAS breakdown

UAS = unlabeled attachment (head only). CLAS = content-word LAS (excludes
function words and punct). Useful for spotting cases where the model has a
decent structure (UAS) but wrong labels (LAS gap), or vice versa.

In [4]:
multi = df[df["metric"].isin(["UAS", "LAS", "CLAS"])]
multi.pivot_table(
    index=["family", "shots", "split"],
    columns="metric",
    values="F1",
).round(2)[["UAS", "LAS", "CLAS"]]

metric                                       UAS    LAS   CLAS
family                       shots split                      
latin-perseus-ud-2.15-241121 0     poetry  59.87  50.78  49.25
                                   prose   64.56  57.68  53.42
latin-perseus-ud-2.17-251125 0     poetry  68.98  61.19  59.90
                                   prose   69.33  62.43  57.46
latinpipe                    0     poetry  78.35  72.27  71.28
                                   prose   80.44  75.06  70.90
my_model                     0     poetry  21.88   1.14   1.24
                                   prose   29.50   0.60   0.69
qwen3-0.6b                   0     poetry  10.93   1.74   2.02
                                   prose   13.36   0.88   0.97
qwen3-0.6b-mlx               0     poetry   9.63   2.67   2.72
                                   prose    7.63   1.62   1.51
                             2     poetry  19.45   2.06   2.23
                                   prose   23.53   1.32   1.52
qwen3-0.6b-mlx.pre-repair    0     poetry  17.35   1.94   1.96
                                   prose   20.81   1.09   1.14
qwen3-vl-8b-instruct-mlx     0     poetry  35.36  18.21  17.42
                                   prose   33.27  17.80  14.53
                             2     poetry  35.90  18.41  17.43
                                   prose   36.46  20.16  16.68

## 5. LAS bar chart

Grouped bars per `(family, k)` across splits — a quick at-a-glance comparison.

In [ ]:
import matplotlib.pyplot as plt

ax = las_table.plot(kind="bar", figsize=(11, 4))
ax.set_ylabel("LAS F1")
ax.set_title("LAS by (family, k-shot) and split")
ax.set_ylim(0, 100)
ax.legend(title="split")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 6. LLMs only — 0-shot vs k-shot side by side

Drop the trained parsers and zoom in on the few-shot comparison. The bar
clusters within each model family put 0-shot directly next to k-shot.

In [ ]:
# Heuristic: a family is an "LLM family" if it has at least one k-shot variant
# OR matches a known LLM family prefix. Adjust this set if you add new LLMs.
llm_families = set(few_shot_families) | {
    f for f in las["family"].unique()
    if any(f.startswith(p) for p in ("qwen", "gemma", "llama", "meta-", "google"))
}

llm_las = las[las["family"].isin(llm_families)]
for split in ("poetry", "prose"):
    sub = llm_las[llm_las["split"] == split].pivot_table(
        index="family", columns="shots", values="F1",
    )
    sub.columns = [f"{int(c)}-shot" for c in sub.columns]
    ax = sub.plot(kind="bar", figsize=(9, 3.5))
    ax.set_ylabel("LAS F1")
    ax.set_title(f"LLM LAS — {split} (0-shot vs k-shot)")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()